In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

# Load full dataset
full_df = spark.read.csv("/Volumes/workspace/default/sales_data/top_customers.csv", header=True, inferSchema=True)

# Write as Delta Table
full_df.write.format("delta").mode("overwrite").save("/Volumes/workspace/default/sales_data/delta_tables/sales_delta")

print("✅ Base Delta table created")


✅ Base Delta table created


In [0]:
inc_df = spark.read.csv("/Volumes/workspace/default/sales_data/top_customers_incremental.csv", header=True, inferSchema=True)


In [0]:
from delta.tables import DeltaTable

delta_table = DeltaTable.forPath(spark, "/Volumes/workspace/default/sales_data/delta_tables/sales_delta")

# Perform incremental merge using 'name' as the key
delta_table.alias("t").merge(
    inc_df.alias("s"),
    "t.name = s.name"   # Matching condition t and s are just alias for target and source
).whenMatchedUpdateAll(
).whenNotMatchedInsertAll(
).execute()

print("✅ Incremental update applied successfully")


✅ Incremental update applied successfully


In [0]:
# Now I have successfully simulated a daily incremental load — just like an Azure Synapse or Data Lake pipeline would handle.
df_updated = spark.read.format("delta").load("/Volumes/workspace/default/sales_data/delta_tables/sales_delta")
df_updated.show(10)
df_updated.count()


+-----------+--------------+
|       Name|Totoal_Revenue|
+-----------+--------------+
|Amit Sharma|        110000|
|  Rahul Das|         80000|
| Kavya Iyer|         20000|
|Rohan Mehta|         15000|
|Priya Singh|         15000|
|Goku Sharma|            10|
|   Binu Das|         54000|
| Sanvi Iyer|         12000|
+-----------+--------------+



8

In [0]:
# Delta table history (versioning)-You can now see every update version — full load, incremental load, etc.
delta_table.history().show(truncate=False)


+-------+-------------------+--------------+----------------------+---------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----+-----------------+------------------------+-----------+-----------------+-------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------